# Use Case 01: Automated Contract Review with LLMs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai/usecases/notebooks/01-contract-review.ipynb)

This notebook implements an end-to-end contract review pipeline that:
1. Extracts text from contract documents
2. Segments the text into individual clauses
3. Classifies each clause by type using embeddings
4. Analyzes each clause against a company playbook using RAG
5. Generates a risk assessment report with severity ratings

**Estimated API cost:** ~$0.04 per full run with GPT-4o

---

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install -q pymupdf chromadb openai tiktoken

In [ ]:
import os
import re
import json
from typing import List, Dict, Optional

# Try to load API key from Colab secrets, fall back to environment variable
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if not OPENAI_API_KEY:
    raise ValueError(
        "Please set your OPENAI_API_KEY. In Colab: use the Secrets panel (key icon). "
        "Locally: export OPENAI_API_KEY='sk-...'"
    )

print("API key loaded successfully.")
print(f"Key prefix: {OPENAI_API_KEY[:8]}...")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

# Configuration
EMBEDDING_MODEL = "text-embedding-3-small"
ANALYSIS_MODEL = "gpt-4o"  # Change to "gpt-4o-mini" to reduce costs ~10x

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Analysis model:  {ANALYSIS_MODEL}")

## 2. Synthetic Contract

We create a realistic NDA / Master Service Agreement with various clause types. This replaces the need for a real contract PDF — the pipeline works identically with extracted PDF text.

In [ ]:
SYNTHETIC_CONTRACT = """
MASTER SERVICE AGREEMENT

This Master Service Agreement ("Agreement") is entered into as of January 15, 2025
(the "Effective Date") by and between:

Party A: TechCorp Solutions Inc., a Delaware corporation ("Client")
Party B: DataFlow Analytics LLC, a California limited liability company ("Provider")

WHEREAS, Client desires to engage Provider for data analytics and AI/ML services, and
Provider desires to provide such services subject to the terms and conditions set forth herein.

NOW, THEREFORE, in consideration of the mutual covenants and agreements hereinafter set forth,
the parties agree as follows:

Section 1. DEFINITIONS

1.1 "Confidential Information" means any non-public information disclosed by either party
to the other party, whether orally, in writing, or by inspection of tangible objects,
including but not limited to trade secrets, business plans, financial information, customer
data, technical data, product plans, and proprietary technology.

1.2 "Deliverables" means all work product, reports, analyses, models, software, and
documentation created by Provider in the performance of the Services.

1.3 "Services" means the data analytics, machine learning model development, and related
consulting services described in each Statement of Work executed under this Agreement.

Section 2. CONFIDENTIALITY

2.1 Each party agrees to hold the other party's Confidential Information in strict confidence
and not to disclose such information to any third party without the prior written consent of
the disclosing party. The receiving party shall protect the disclosing party's Confidential
Information using the same degree of care it uses to protect its own confidential information,
but in no event less than reasonable care.

2.2 The obligations of confidentiality shall survive termination of this Agreement for a
period of five (5) years from the date of disclosure.

2.3 Confidential Information shall not include information that: (a) is or becomes publicly
available through no fault of the receiving party; (b) was known to the receiving party prior
to disclosure; (c) is independently developed by the receiving party; or (d) is rightfully
received from a third party without restriction.

Section 3. INTELLECTUAL PROPERTY ASSIGNMENT

3.1 All Deliverables created by Provider under this Agreement shall be considered "work
made for hire" and shall be the sole and exclusive property of Provider. Client is granted
a non-exclusive, non-transferable license to use the Deliverables solely for its internal
business purposes.

3.2 Provider retains all rights, title, and interest in and to any pre-existing intellectual
property, tools, frameworks, and methodologies used in the creation of Deliverables
("Provider IP"). Client receives no rights to Provider IP except as expressly stated herein.

3.3 Any improvements, modifications, or derivative works of Provider IP created during
the performance of Services shall be the exclusive property of Provider.

Section 4. INDEMNIFICATION

4.1 Client shall indemnify, defend, and hold harmless Provider and its officers, directors,
employees, and agents from and against any and all claims, damages, losses, liabilities,
costs, and expenses (including reasonable attorneys' fees) arising out of or relating to:
(a) Client's use of the Deliverables; (b) Client's breach of this Agreement; (c) any claim
that Client's data or materials infringe the intellectual property rights of any third party.

4.2 Provider shall have no obligation to indemnify Client under any circumstances. This
indemnification is strictly one-directional in favor of Provider.

Section 5. LIMITATION OF LIABILITY

5.1 IN NO EVENT SHALL PROVIDER BE LIABLE TO CLIENT FOR ANY INDIRECT, INCIDENTAL,
SPECIAL, CONSEQUENTIAL, OR PUNITIVE DAMAGES, REGARDLESS OF THE CAUSE OF ACTION OR THE
THEORY OF LIABILITY, EVEN IF PROVIDER HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.

5.2 PROVIDER'S TOTAL AGGREGATE LIABILITY ARISING OUT OF OR RELATED TO THIS AGREEMENT
SHALL NOT EXCEED THE FEES PAID BY CLIENT TO PROVIDER IN THE THREE (3) MONTHS PRECEDING
THE EVENT GIVING RISE TO THE CLAIM.

5.3 THE FOREGOING LIMITATIONS SHALL APPLY NOTWITHSTANDING ANY FAILURE OF ESSENTIAL
PURPOSE OF ANY LIMITED REMEDY.

Section 6. TERMINATION

6.1 Either party may terminate this Agreement for convenience upon thirty (30) days' prior
written notice to the other party.

6.2 Either party may terminate this Agreement immediately upon written notice if the other
party: (a) materially breaches this Agreement and fails to cure such breach within fifteen
(15) days after receiving written notice thereof; or (b) becomes insolvent, files for
bankruptcy, or ceases to conduct business in the normal course.

6.3 Upon termination, Client shall pay Provider for all Services rendered and expenses
incurred through the effective date of termination. All provisions that by their nature
should survive termination shall survive, including Sections 2, 3, 4, 5, and 8.

Section 7. NON-COMPETE AND NON-SOLICITATION

7.1 During the term of this Agreement and for a period of twenty-four (24) months following
termination, Client shall not, directly or indirectly, engage in or assist any business that
competes with Provider's data analytics and AI/ML services within North America.

7.2 During the term and for twelve (12) months after termination, neither party shall
solicit or hire any employee of the other party who was involved in the performance of
Services under this Agreement.

Section 8. GOVERNING LAW AND DISPUTE RESOLUTION

8.1 This Agreement shall be governed by and construed in accordance with the laws of the
State of Delaware, without regard to its conflict of laws principles.

8.2 Any dispute arising out of or relating to this Agreement shall be resolved exclusively
through binding arbitration administered by the American Arbitration Association in
Wilmington, Delaware. The arbitrator's decision shall be final and binding.

8.3 The prevailing party in any arbitration or legal proceeding shall be entitled to recover
its reasonable attorneys' fees and costs from the non-prevailing party.

Section 9. DATA PRIVACY

9.1 To the extent Provider processes any personal data on behalf of Client, Provider agrees
to process such data solely in accordance with Client's documented instructions.

9.2 Provider shall implement appropriate technical and organizational measures to ensure a
level of security appropriate to the risk of processing personal data.

9.3 Provider may transfer personal data to sub-processors located outside the European
Economic Area without Client's prior consent, provided that Provider ensures adequate
data protection measures are in place.

Section 10. AUTO-RENEWAL AND TERM

10.1 The initial term of this Agreement is one (1) year from the Effective Date. Thereafter,
this Agreement shall automatically renew for successive one-year periods unless either party
provides written notice of non-renewal at least ninety (90) days prior to the end of the
then-current term.

10.2 Provider reserves the right to increase fees by up to fifteen percent (15%) upon each
renewal, with written notice to Client at least sixty (60) days prior to the renewal date.

IN WITNESS WHEREOF, the parties have executed this Agreement as of the Effective Date.
"""

print(f"Contract length: {len(SYNTHETIC_CONTRACT):,} characters")
print(f"Approximate tokens: ~{len(SYNTHETIC_CONTRACT) // 4:,}")

## 3. Clause Extraction & Classification

We use regex patterns to segment the contract into individual clauses, then classify each one.

In [ ]:
# Regex patterns for clause boundary detection
CLAUSE_PATTERNS = [
    re.compile(r'^(?:Section|Article|Clause)\s+\d+[\.]?\s*[A-Z]', re.MULTILINE),
    re.compile(r'^\d+\.\d+\s', re.MULTILINE),
]

# Pattern to identify section-level boundaries
SECTION_PATTERN = re.compile(
    r'^Section\s+(\d+)\.\s+([A-Z][A-Z\s&]+)',
    re.MULTILINE
)


def segment_by_section(text: str) -> List[Dict]:
    """Split contract text into sections based on 'Section N. TITLE' pattern."""
    matches = list(SECTION_PATTERN.finditer(text))

    if not matches:
        return [{"index": 0, "title": "Full Contract", "text": text.strip()}]

    sections = []
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_text = text[start:end].strip()
        section_num = match.group(1)
        section_title = match.group(2).strip()

        sections.append({
            "index": i,
            "section_num": section_num,
            "title": section_title,
            "text": section_text,
            "char_start": start,
            "char_end": end,
        })

    return sections


# Segment the contract
clauses = segment_by_section(SYNTHETIC_CONTRACT)

print(f"Found {len(clauses)} sections:\n")
for clause in clauses:
    print(f"  Section {clause['section_num']}: {clause['title']}")
    print(f"    Length: {len(clause['text'])} chars")
    print()

In [ ]:
# Clause type mapping based on section titles
CLAUSE_TYPE_MAP = {
    "DEFINITIONS": "definitions",
    "CONFIDENTIALITY": "confidentiality",
    "INTELLECTUAL PROPERTY ASSIGNMENT": "ip_assignment",
    "INDEMNIFICATION": "indemnification",
    "LIMITATION OF LIABILITY": "liability_cap",
    "TERMINATION": "termination",
    "NON-COMPETE AND NON-SOLICITATION": "non_compete",
    "GOVERNING LAW AND DISPUTE RESOLUTION": "governing_law",
    "DATA PRIVACY": "data_privacy",
    "AUTO-RENEWAL AND TERM": "auto_renewal",
}


def classify_clause(title: str) -> str:
    """Map section title to clause type. Falls back to 'other'."""
    return CLAUSE_TYPE_MAP.get(title.strip(), "other")


# Add clause types to our segmented clauses
for clause in clauses:
    clause["clause_type"] = classify_clause(clause["title"])

print("Clause classifications:\n")
for clause in clauses:
    print(f"  {clause['title']:45s} -> {clause['clause_type']}")

## 4. Build the Playbook Vector Store

The company playbook contains guidelines for each clause type — what's acceptable, what's a red flag, and market standard benchmarks. We store it in ChromaDB for RAG retrieval.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize ChromaDB with OpenAI embeddings
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=EMBEDDING_MODEL
)

chroma_client = chromadb.Client()

# Delete existing collection if it exists (for re-runs)
try:
    chroma_client.delete_collection("contract_playbook")
except Exception:
    pass

playbook_collection = chroma_client.create_collection(
    name="contract_playbook",
    embedding_function=openai_ef,
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created.")

In [ ]:
# Company playbook entries — guidelines for each clause type
PLAYBOOK_ENTRIES = [
    {
        "id": "confidentiality_guide",
        "clause_type": "confidentiality",
        "text": """CONFIDENTIALITY GUIDELINES:
Acceptable: Mutual confidentiality obligations with standard exceptions (public knowledge,
prior knowledge, independent development, third-party disclosure). Survival period of 2-5 years.
Red Flags: One-sided confidentiality favoring counterparty only. Survival period exceeding
7 years. No exceptions to confidentiality obligations. Obligations extending to affiliates
without limitation.
Market Standard: Mutual NDA with 3-year survival. Standard carve-outs. Reasonable care standard
(not strict liability for breaches)."""
    },
    {
        "id": "ip_assignment_guide",
        "clause_type": "ip_assignment",
        "text": """IP ASSIGNMENT GUIDELINES:
Acceptable: Client owns all custom deliverables created specifically for Client. Provider retains
pre-existing IP with license to Client. Joint IP provisions for collaborative work.
Red Flags: Provider retains ownership of all deliverables (Client only gets a license).
Work-for-hire doctrine claimed by provider. No license to underlying tools or frameworks.
No rights to derivative works based on Client data.
Market Standard: Client owns custom deliverables. Provider retains pre-existing IP and tools
with perpetual, royalty-free license to Client. Clear definition of what constitutes custom
vs. pre-existing IP."""
    },
    {
        "id": "indemnification_guide",
        "clause_type": "indemnification",
        "text": """INDEMNIFICATION GUIDELINES:
Acceptable: Mutual indemnification covering IP infringement and material breach. Each party
indemnifies the other proportional to fault. Indemnity capped at contract value or insurance limits.
Red Flags: Unlimited one-sided indemnification against Client. No mutual indemnity obligation.
Provider has zero indemnification obligations. Client indemnifies Provider for Provider's own
negligence or willful misconduct.
Market Standard: Mutual indemnification with caps at 1-2x annual contract value. Each party
covers its own breaches. IP indemnity from Provider for deliverables."""
    },
    {
        "id": "liability_cap_guide",
        "clause_type": "liability_cap",
        "text": """LIABILITY CAP GUIDELINES:
Acceptable: Mutual liability cap at 1-2x annual contract value. Exclusions for gross negligence,
willful misconduct, and IP infringement. Consequential damages waiver is mutual.
Red Flags: Liability cap below fees paid in the last 3 months. One-sided cap (Provider capped,
Client uncapped). No exclusions from the cap. Consequential damages waiver only benefits Provider.
Uncapped liability for Client's indemnification obligations.
Market Standard: Cap at 12 months of fees. Mutual consequential damages waiver. Carve-outs for
confidentiality breach and IP infringement (2-3x higher cap). Gross negligence and willful
misconduct excluded from cap."""
    },
    {
        "id": "termination_guide",
        "clause_type": "termination",
        "text": """TERMINATION GUIDELINES:
Acceptable: Mutual termination for convenience with 30-60 day notice. Termination for cause with
reasonable cure period (30 days). Clear wind-down provisions. Transition assistance period.
Red Flags: Termination penalties or early termination fees. Cure period less than 15 days.
No termination for convenience right. Provider can terminate without cause on short notice.
Survival clauses that are overly broad.
Market Standard: Mutual 30-day convenience termination. 30-day cure for material breach.
Payment for work completed through termination. 90-day transition assistance."""
    },
    {
        "id": "non_compete_guide",
        "clause_type": "non_compete",
        "text": """NON-COMPETE GUIDELINES:
Acceptable: Non-solicitation of employees for 12 months. Narrow non-compete limited to specific
services and geography for up to 12 months.
Red Flags: Non-compete lasting more than 12 months. Broad geographic scope (national or global).
Non-compete restricting Client's core business activities. One-sided non-compete only binding
on Client. Non-compete on the Client (our company) preventing us from engaging competitors.
Market Standard: Mutual non-solicitation of employees for 12 months. No non-compete on Client.
If non-compete exists, limited to 6-12 months and narrowly defined services/geography."""
    },
    {
        "id": "governing_law_guide",
        "clause_type": "governing_law",
        "text": """GOVERNING LAW GUIDELINES:
Acceptable: Delaware, New York, or Client's home state law. Dispute resolution via arbitration
or courts in a neutral or Client-preferred jurisdiction.
Red Flags: Foreign jurisdiction governing law (unless international deal). Mandatory arbitration
in Provider's home jurisdiction with no appeal rights. Waiver of jury trial without mutual benefit.
Market Standard: Delaware or New York law. AAA or JAMS arbitration. Prevailing party attorney
fees provision."""
    },
    {
        "id": "data_privacy_guide",
        "clause_type": "data_privacy",
        "text": """DATA PRIVACY GUIDELINES:
Acceptable: Provider processes data per Client instructions only. Appropriate security measures
required. Sub-processor approval required from Client. Data breach notification within 72 hours.
GDPR/CCPA compliance commitments. Data deletion upon termination.
Red Flags: Provider can transfer data to sub-processors without Client consent. No data breach
notification timeline. No GDPR compliance commitments for EU data. No data deletion obligations.
Provider retains rights to use Client data for its own purposes.
Market Standard: Full GDPR Article 28 compliance. 72-hour breach notification. Prior written
consent for sub-processors. Annual security audits. Data returned/deleted within 30 days of
termination."""
    },
    {
        "id": "auto_renewal_guide",
        "clause_type": "auto_renewal",
        "text": """AUTO-RENEWAL GUIDELINES:
Acceptable: Auto-renewal with 30-60 day opt-out notice. Fee increases capped at CPI or 3-5%.
Clear notice mechanism. Renewal on same terms unless mutually agreed.
Red Flags: Auto-renewal requiring more than 90 days notice to cancel. Fee increases above 10%.
Multi-year renewal periods. No cap on fee increases. Evergreen clause with no termination
mechanism.
Market Standard: Annual auto-renewal. 30-60 day opt-out. Fee increases capped at 5% or CPI,
whichever is lower. Written notice via email acceptable."""
    },
]

# 💰 Cost Warning: Embedding API calls for playbook entries
playbook_collection.add(
    ids=[e["id"] for e in PLAYBOOK_ENTRIES],
    documents=[e["text"] for e in PLAYBOOK_ENTRIES],
    metadatas=[{"clause_type": e["clause_type"]} for e in PLAYBOOK_ENTRIES],
)

print(f"Added {len(PLAYBOOK_ENTRIES)} playbook entries to ChromaDB.")
print(f"Collection count: {playbook_collection.count()}")

## 5. RAG-Based Clause Analysis

For each clause, we retrieve relevant playbook guidance and ask the LLM to produce a structured risk analysis.

In [ ]:
ANALYSIS_SYSTEM_PROMPT = """You are an expert contract attorney AI assistant.
You analyze individual contract clauses against company guidelines.

For each clause, provide a structured analysis in JSON format with these exact keys:
{
  "clause_type": "the type of clause",
  "summary": "1-2 sentence plain-English summary of what this clause says",
  "risk_level": "CRITICAL | HIGH | MEDIUM | LOW",
  "risk_score": <integer 0-100, where 0 is no risk and 100 is maximum risk>,
  "findings": [
    {
      "issue": "specific issue identified",
      "explanation": "why this is a concern in plain language",
      "playbook_deviation": "how it deviates from company guidelines",
      "recommendation": "specific action to take"
    }
  ],
  "missing_protections": ["list of standard protections that are absent"],
  "negotiation_points": ["specific changes to request from counterparty"]
}

Rules:
- Be precise and cite specific language from the clause text.
- Compare against the playbook guidelines provided in the context.
- If the clause is acceptable per the playbook, assign LOW risk and say so clearly.
- Use CRITICAL for issues that could cause major financial or legal exposure.
- Use HIGH for significant deviations from company standards.
- Use MEDIUM for issues worth addressing but not deal-breakers.
- Use LOW for minor concerns or fully acceptable clauses.
- Always return valid JSON."""

print("Analysis system prompt configured.")
print(f"Prompt length: {len(ANALYSIS_SYSTEM_PROMPT)} chars")

In [ ]:
def analyze_clause(clause: Dict, collection, openai_client) -> Dict:
    """Analyze a single clause against the company playbook using RAG."""
    clause_type = clause["clause_type"]
    clause_text = clause["text"]

    # Skip definitions section — low risk, no playbook entry
    if clause_type == "definitions":
        return {
            "clause_type": "definitions",
            "summary": "Standard definitions section defining key terms used in the agreement.",
            "risk_level": "LOW",
            "risk_score": 5,
            "findings": [],
            "missing_protections": [],
            "negotiation_points": [],
        }

    # Step 1: Retrieve relevant playbook guidance via RAG
    # 💰 Cost Warning: Embedding API call for query
    try:
        results = collection.query(
            query_texts=[clause_text],
            n_results=2,
            where={"clause_type": clause_type} if clause_type != "other" else None,
        )
        playbook_context = "\n\n".join(results["documents"][0])
    except Exception:
        # Fallback: query without filter
        results = collection.query(
            query_texts=[clause_text],
            n_results=2,
        )
        playbook_context = "\n\n".join(results["documents"][0])

    # Step 2: Build the analysis prompt
    user_prompt = f"""Analyze this contract clause against our company playbook guidelines.

CLAUSE TITLE: {clause.get('title', 'Unknown')}
CLAUSE TYPE: {clause_type}

CLAUSE TEXT:
{clause_text}

COMPANY PLAYBOOK GUIDANCE:
{playbook_context}

Provide your structured analysis as a JSON object."""

    # Step 3: Call the LLM
    # 💰 Cost Warning: LLM API call (~$0.005 per clause with GPT-4o)
    response = openai_client.chat.completions.create(
        model=ANALYSIS_MODEL,
        messages=[
            {"role": "system", "content": ANALYSIS_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
    )

    analysis = json.loads(response.choices[0].message.content)
    return analysis

print("analyze_clause function defined.")

In [ ]:
# 💰 Cost Warning: This cell runs LLM analysis on all clauses
# Estimated cost: ~$0.04 for 9 clauses with GPT-4o

print("Analyzing clauses against company playbook...\n")
print("=" * 60)

clause_analyses = []

for clause in clauses:
    print(f"\nAnalyzing: Section {clause['section_num']} - {clause['title']}")
    print(f"  Type: {clause['clause_type']}")

    analysis = analyze_clause(clause, playbook_collection, client)
    clause_analyses.append(analysis)

    risk_level = analysis.get("risk_level", "N/A")
    risk_score = analysis.get("risk_score", "N/A")
    num_findings = len(analysis.get("findings", []))

    # Color-code the risk level for display
    risk_indicator = {
        "CRITICAL": "🔴",
        "HIGH": "🟠",
        "MEDIUM": "🟡",
        "LOW": "🟢",
    }.get(risk_level, "⚪")

    print(f"  Result: {risk_indicator} {risk_level} (score: {risk_score}/100)")
    print(f"  Findings: {num_findings} issue(s) identified")

print("\n" + "=" * 60)
print(f"\nAnalysis complete. {len(clause_analyses)} clauses processed.")

## 6. Risk Scoring & Report Generation

We aggregate individual clause analyses into an overall risk assessment using weighted scoring.

In [ ]:
# Clause type importance weights
CLAUSE_WEIGHTS = {
    "indemnification": 1.5,
    "liability_cap": 1.5,
    "ip_assignment": 1.4,
    "change_of_control": 1.3,
    "termination": 1.2,
    "non_compete": 1.2,
    "data_privacy": 1.1,
    "confidentiality": 1.0,
    "governing_law": 0.8,
    "force_majeure": 0.7,
    "auto_renewal": 0.6,
    "definitions": 0.2,
    "other": 0.5,
}


def compute_overall_risk(analyses: List[Dict]) -> Dict:
    """Compute weighted overall risk score from clause analyses."""
    total_weighted_score = 0
    total_weight = 0
    critical_count = 0
    high_count = 0
    medium_count = 0
    low_count = 0

    for analysis in analyses:
        clause_type = analysis.get("clause_type", "other")
        risk_score = analysis.get("risk_score", 50)
        weight = CLAUSE_WEIGHTS.get(clause_type, 0.5)

        total_weighted_score += risk_score * weight
        total_weight += weight

        level = analysis.get("risk_level", "MEDIUM")
        if level == "CRITICAL":
            critical_count += 1
        elif level == "HIGH":
            high_count += 1
        elif level == "MEDIUM":
            medium_count += 1
        else:
            low_count += 1

    overall_score = total_weighted_score / total_weight if total_weight > 0 else 0

    if critical_count > 0 or overall_score >= 75:
        verdict = "REJECT — Requires significant negotiation before signing"
    elif high_count > 2 or overall_score >= 50:
        verdict = "REVIEW — Multiple issues need attention before proceeding"
    elif overall_score >= 25:
        verdict = "CONDITIONAL — Minor issues to address in negotiation"
    else:
        verdict = "APPROVE — Within acceptable parameters"

    return {
        "overall_risk_score": round(overall_score, 1),
        "verdict": verdict,
        "critical_issues": critical_count,
        "high_issues": high_count,
        "medium_issues": medium_count,
        "low_issues": low_count,
        "total_clauses": len(analyses),
        "clause_analyses": analyses,
    }


# Compute the overall risk assessment
risk_assessment = compute_overall_risk(clause_analyses)

print("Overall Risk Assessment")
print("=" * 40)
print(f"Risk Score:      {risk_assessment['overall_risk_score']}/100")
print(f"Verdict:         {risk_assessment['verdict']}")
print(f"Critical Issues: {risk_assessment['critical_issues']}")
print(f"High Issues:     {risk_assessment['high_issues']}")
print(f"Medium Issues:   {risk_assessment['medium_issues']}")
print(f"Low Issues:      {risk_assessment['low_issues']}")
print(f"Total Clauses:   {risk_assessment['total_clauses']}")

In [ ]:
def generate_report(assessment: Dict) -> str:
    """Generate a human-readable risk report."""
    lines = [
        "=" * 70,
        "         CONTRACT RISK ASSESSMENT REPORT",
        "=" * 70,
        "",
        f"  Overall Risk Score:  {assessment['overall_risk_score']}/100",
        f"  Verdict:             {assessment['verdict']}",
        f"  Critical Issues:     {assessment['critical_issues']}",
        f"  High Issues:         {assessment['high_issues']}",
        f"  Medium Issues:       {assessment['medium_issues']}",
        f"  Low Issues:          {assessment['low_issues']}",
        f"  Total Clauses:       {assessment['total_clauses']}",
        "",
        "-" * 70,
        "  CLAUSE-BY-CLAUSE BREAKDOWN",
        "-" * 70,
    ]

    for i, analysis in enumerate(assessment["clause_analyses"], 1):
        risk_indicator = {
            "CRITICAL": "[!!!]",
            "HIGH": "[!! ]",
            "MEDIUM": "[!  ]",
            "LOW": "[ OK]",
        }.get(analysis.get("risk_level", "N/A"), "[   ]")

        clause_type = analysis.get("clause_type", "unknown").upper()
        risk_level = analysis.get("risk_level", "N/A")
        risk_score = analysis.get("risk_score", "N/A")
        summary = analysis.get("summary", "N/A")

        lines.append(f"")
        lines.append(f"  {risk_indicator} [{i}] {clause_type}")
        lines.append(f"       Risk Level: {risk_level} ({risk_score}/100)")
        lines.append(f"       Summary: {summary}")

        findings = analysis.get("findings", [])
        if findings:
            lines.append(f"       Findings:")
            for j, finding in enumerate(findings, 1):
                lines.append(f"         {j}. {finding.get('issue', 'N/A')}")
                lines.append(f"            Why: {finding.get('explanation', 'N/A')}")
                lines.append(f"            Fix: {finding.get('recommendation', 'N/A')}")

        negotiation_points = analysis.get("negotiation_points", [])
        if negotiation_points:
            lines.append(f"       Negotiation Points:")
            for point in negotiation_points:
                lines.append(f"         - {point}")

    lines.append("")
    lines.append("=" * 70)
    lines.append("  END OF REPORT")
    lines.append("=" * 70)

    return "\n".join(lines)


# Generate and display the report
report = generate_report(risk_assessment)
print(report)

## 7. Detailed Findings Viewer

Let's look at the high-risk clauses in detail.

In [ ]:
# Display detailed analysis for high-risk clauses
print("HIGH-RISK CLAUSE DETAILS")
print("=" * 60)

high_risk = [
    a for a in clause_analyses
    if a.get("risk_level") in ("CRITICAL", "HIGH")
]

if not high_risk:
    print("\nNo CRITICAL or HIGH risk clauses found.")
else:
    for analysis in high_risk:
        print(f"\n{'─' * 60}")
        print(f"Clause Type:  {analysis.get('clause_type', 'N/A').upper()}")
        print(f"Risk Level:   {analysis.get('risk_level', 'N/A')}")
        print(f"Risk Score:   {analysis.get('risk_score', 'N/A')}/100")
        print(f"Summary:      {analysis.get('summary', 'N/A')}")

        print(f"\n  Findings:")
        for j, finding in enumerate(analysis.get("findings", []), 1):
            print(f"    {j}. Issue: {finding.get('issue', 'N/A')}")
            print(f"       Explanation: {finding.get('explanation', 'N/A')}")
            print(f"       Playbook Deviation: {finding.get('playbook_deviation', 'N/A')}")
            print(f"       Recommendation: {finding.get('recommendation', 'N/A')}")
            print()

        missing = analysis.get("missing_protections", [])
        if missing:
            print(f"  Missing Protections:")
            for item in missing:
                print(f"    - {item}")

        negotiation = analysis.get("negotiation_points", [])
        if negotiation:
            print(f"  Negotiation Points:")
            for item in negotiation:
                print(f"    - {item}")

print(f"\n{'═' * 60}")
print(f"Total high-risk clauses: {len(high_risk)} out of {len(clause_analyses)}")

## 8. Export Results

Save the full analysis as JSON for integration with CLM systems or further processing.

In [ ]:
# Export the full analysis as JSON
output = {
    "contract_name": "Master Service Agreement — TechCorp Solutions / DataFlow Analytics",
    "analysis_model": ANALYSIS_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "overall_risk_score": risk_assessment["overall_risk_score"],
    "verdict": risk_assessment["verdict"],
    "summary": {
        "critical": risk_assessment["critical_issues"],
        "high": risk_assessment["high_issues"],
        "medium": risk_assessment["medium_issues"],
        "low": risk_assessment["low_issues"],
        "total": risk_assessment["total_clauses"],
    },
    "clause_analyses": clause_analyses,
}

output_path = "contract_analysis_report.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Full analysis exported to: {output_path}")
print(f"File size: {os.path.getsize(output_path):,} bytes")

# Quick preview
print(f"\nJSON structure:")
print(json.dumps({k: type(v).__name__ for k, v in output.items()}, indent=2))

## 9. Summary & Next Steps

This notebook demonstrated the full automated contract review pipeline:

| Step | What We Did | Technology |
|------|-------------|------------|
| Text Extraction | Used synthetic contract (production: PyMuPDF) | PyMuPDF / fitz |
| Clause Segmentation | Regex-based section splitting | Python re module |
| Classification | Title-based mapping (production: embedding similarity) | ChromaDB |
| RAG Analysis | Retrieved playbook guidance per clause type | ChromaDB + OpenAI |
| Risk Scoring | Weighted formula with severity thresholds | Custom Python |
| Report Generation | Structured text + JSON export | Python |

### To extend this project:
- **Add PDF support**: Replace the synthetic contract with `fitz.open("contract.pdf")` for real PDFs
- **Add OCR**: Use Tesseract for scanned documents
- **Expand the playbook**: Add more clause types and jurisdiction-specific guidelines
- **Try different models**: Swap `gpt-4o` for `gpt-4o-mini` (cheaper) or `claude-3-5-sonnet` (different perspective)
- **Add a feedback loop**: Track attorney corrections to improve scoring thresholds over time
- **Batch processing**: Use `asyncio` with the OpenAI async client for parallel clause analysis

---

**Study guide**: [CareerAlign GenAI Use Cases — Automated Contract Review](https://careeralign.com/genai/usecases/01-contract-review.html)